# 4) Gender in the Text: Pronouns & Nearby Verbs

**Goal:** Compare relative frequency of pronouns and the verbs near them.

# Setup: Load Texts

This notebook needs **Crime And Punishment** and **The Brothers Karamazov** as input texts.

**How to provide the texts:**
1. Download books from Project Gutenberg (IDs 11 and 12) as txts. [go to https://www.gutenberg.org/ebooks/11 and https://www.gutenberg.org/ebooks/12]

2. Place two text files in the "data" folder with names:
   - `Crime-punishment.txt`  (Crime And Punishment)
   - `The-BrothersKaramazov.txt` (The Brothers Karamazov)

In [1]:
import re
from pathlib import Path

In [2]:

def load_texts(local_alice: str = '../data/Crime-punishment.txt',
               local_glass: str = '../data/The-BrothersKaramazov.txt'):
    """Load Wonderland and Looking-Glass texts from disk.

    Parameters
    ----------
    local_alice : str
        Path to Wonderland text file. Defaults to '../data/Crime-punishment.txt'.
    local_glass : str
        Path to Looking-Glass text file. Defaults to '../data/The-BrothersKaramazov.txt'.

    Returns
    -------
    tuple[str, str]
        (wonderland_text, lookingglass_text).

    Raises
    ------
    FileNotFoundError
        If either file is missing.

    Extra Notes
    -----------
    - Using UTF-8 with `errors='ignore'` avoids codec exceptions on
      older Project Gutenberg dumps or inconsistent encodings.
    """
    p1, p2 = Path(local_alice), Path(local_glass)

    # Fail fast with a clear message if a file is missing
    if not p1.exists():
        raise FileNotFoundError(
            f"Missing file: {p1}\n"
            "→ Please place 'Crime-punishment.txt' at this path or update load_texts(...)."
        )
    if not p2.exists():
        raise FileNotFoundError(
            f"Missing file: {p2}\n"
            "→ Please place 'The-BrothersKaramazov.txt' at this path or update load_texts(...)."
        )

    # Read the files (UTF-8; ignore undecodable bytes to stay robust)
    wonderland   = p1.read_text(encoding='utf-8', errors='ignore')
    lookingglass = p2.read_text(encoding='utf-8', errors='ignore')
    return wonderland, lookingglass

def normalize(text: str) -> str:
    """Normalize a Gutenberg-like text for tokenization.

    Steps
    -----
    1) Heuristically strip Project Gutenberg headers/footers if present
       (looks for *** START ... *** END markers).
    2) Normalize newlines to '\n'.

    Parameters
    ----------
    text : str
        Raw text as loaded from disk (can be empty).

    Returns
    -------
    str
        Cleaned text suitable for tokenization and counting.
    """
    if not text:
        return ''
    # Clip to the main body if markers are present.
    start = text.find('*** START')
    end   = text.find('*** END')
    if start != -1 and end != -1 and end > start:
        text = text[start:end]
    # Normalize Windows line endings.
    return text.replace('\r\n', '\n')

# Load raw texts (forgiving: returns '' if a file is missing)
CrimePunishment_raw, TheBrothers_raw = load_texts()

# Normalize for tokenization
CrimePunishment   = normalize(CrimePunishment_raw)
TheBrothers  = normalize(TheBrothers_raw)

print(f"Crime&Punishment chars: {len(CrimePunishment):,} | The-brothers-Karamazov chars: {len(TheBrothers):,}")


Crime&Punishment chars: 1,224,432 | The-brothers-Karamazov chars: 1,956,247


### Helpers: Tokenization

In [3]:
WORD_RE = re.compile(r"[A-Za-z']+")  # keep apostrophes in words (e.g., don't -> don't)

def words(text: str):
    """Simple word tokenizer (lowercased, ASCII letters + apostrophes).

    Pros
    ----
    - Very fast and dependency-free.
    - Good enough for frequency/keyness demonstrations.

    Cons
    ----
    - No punctuation words, no sentence boundaries, no POS tags.
    - May treat possessives inconsistently across sources.

    Returns
    -------
    list[str]
        Lowercased word words.
    """
    return WORD_RE.findall(text.lower())


def sentences(text: str):
    """Naive sentence splitter using punctuation boundaries.

    Uses a regex to split on '.', '!', '?' followed by whitespace.
    Because this is heuristic, treat results as approximate.

    Returns
    -------
    list[str]
        Sentence-like strings.
    """
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]



CrimePunishment_words = words(CrimePunishment)
TheBrothers_words = words(TheBrothers)

CrimePunishment_sentences = sentences(CrimePunishment)
TheBrothers_sentences = sentences(TheBrothers)

print(f"CrimePunishment words: {len(CrimePunishment_words):,} | TheBrothers words: {len(TheBrothers_words):,}")
print(f"CrimePunishment sentences: {len(CrimePunishment_sentences):,} | TheBrothers sentences: {len(TheBrothers_sentences):,}")


CrimePunishment words: 214,498 | TheBrothers words: 359,146
CrimePunishment sentences: 16,994 | TheBrothers sentences: 19,234


### Pronoun Balance

In [4]:

from collections import Counter
def pronoun_counts(tokens):
    target = {'he','she','him','her'}
    c = Counter(w for w in tokens if w in target)
    total = sum(c.values())
    return c, total

a_c, a_tot = pronoun_counts(CrimePunishment_words)
g_c, g_tot = pronoun_counts(TheBrothers_words)
print("CrimePunishment words:", dict(a_c), "total:", a_tot)
print("TheBrothers words:", dict(g_c), "total:", g_tot)


CrimePunishment words: {'he': 4903, 'him': 1613, 'she': 1697, 'her': 1841} total: 10054
TheBrothers words: {'him': 3377, 'he': 8149, 'she': 1784, 'her': 1885} total: 15195


### Verbs Near Pronouns (very naive)

In [5]:

def verb_like(word):
    # crude heuristic: words ending in common verb suffixes or base forms
    return bool(re.match(r".*(ed|ing|s)$", word)) or word in {"say","says","said","go","goes","went","come","comes","came","think","thinks","thought","see","sees","saw","know","knows","knew"}

def verbs_near_pronouns(tokens, window=2):
    verbs_for = {'he':[], 'she':[]}
    for i,w in enumerate(tokens):
        if w in ('he','she'):
            for j in range(max(0,i-window), min(len(tokens), i+window+1)):
                if j==i: continue
                if verb_like(tokens[j]):
                    verbs_for[w].append(tokens[j])
    return {k: Counter(v).most_common(20) for k,v in verbs_for.items()}

print("CrimePunishment :", verbs_near_pronouns(CrimePunishment_words) )
print("TheBrothers:", verbs_near_pronouns(TheBrothers_words
) )

CrimePunishment : {'he': [('was', 582), ('is', 253), ('his', 209), ('as', 206), ('said', 186), ('went', 131), ('s', 122), ('thought', 116), ('has', 98), ('looked', 76), ('asked', 57), ('added', 57), ('turned', 57), ('come', 56), ('cried', 52), ('saw', 51), ('this', 50), ('knew', 49), ('walked', 42), ('say', 39)], 'she': [('was', 240), ('is', 180), ('as', 65), ('has', 62), ('said', 60), ('s', 45), ('cried', 32), ('looked', 26), ('came', 23), ('knew', 22), ('does', 21), ('went', 20), ('turned', 19), ('know', 18), ('asked', 18), ('nothing', 15), ('come', 15), ('this', 14), ('seemed', 14), ('added', 14)]}
TheBrothers: {'he': [('was', 1161), ('is', 446), ('s', 375), ('his', 371), ('as', 354), ('said', 302), ('has', 180), ('went', 117), ('thought', 102), ('cried', 98), ('knew', 96), ('looked', 93), ('know', 82), ('this', 80), ('saw', 73), ('came', 71), ('say', 70), ('come', 68), ('added', 65), ('asked', 64)], 'she': [('was', 259), ('is', 135), ('s', 113), ('said', 70), ('as', 65), ('has', 49

**Prompt:** How do these crude patterns line up with character agency and narrative voice? What errors do you notice, and how would POS (Part OF Speech) tagging improve this?

In [6]:
'''

-----------------------------------------
1. Overview
-----------------------------------------
The analysis compares two books — Crime and Punishment and The Brothers Karamazov — by examining which words most frequently follow “he” and “she.” 
This gives a crude sense of how male and female characters are described or portrayed in the narrative.

Example data:
Crime and Punishment (he): was, is, his, said, went, thought, asked...
Crime and Punishment (she): was, is, said, cried, looked, seemed...
The Brothers Karamazov (he): was, is, said, went, thought, cried...
The Brothers Karamazov (she): was, is, said, cried, loved, looked...

-----------------------------------------
2. Character Agency and Narrative Voice
-----------------------------------------
Character agency refers to how active or passive characters are — whether they perform actions or have actions performed on them.
Narrative voice refers to how the narrator describes and frames the characters.

From the data:
- “He” is frequently followed by verbs of action or thought (went, thought, said, looked, asked).
- “She” is often followed by verbs or structures that express states or emotion (was, seemed, cried, loved).

Interpretation:
- Male characters tend to act, think, and speak — suggesting higher agency.
- Female characters are more often described or emotionally expressed — suggesting lower agency or a more external narrative focus.

This reflects a narrative imbalance: men “do,” women “are.”

-----------------------------------------
3. Limitations and Errors in the Crude Method
-----------------------------------------
This bigram-based analysis is surface-level and lacks grammatical understanding. Some limitations include:

1. Ambiguity of pronoun reference:
   - “He” or “she” might not always refer to the same character or even a person.

2. Part-of-speech (POS) ambiguity:
   - Words like “was” or “s” (from “he’s”) have different grammatical roles.

3. Context ignored:
   - “He was angry” vs. “He was hit” are treated the same, even though one is active, one is passive.

4. No sentence-level understanding:
   - The analysis can’t distinguish subjects from objects or identify who performs an action.

-----------------------------------------
4. How POS Tagging Improves the Analysis
-----------------------------------------
Part-of-Speech (POS) tagging labels each word by its grammatical function (e.g., noun, verb, adjective). 
It helps reveal structure, meaning, and agency more accurately.

Benefits of POS Tagging:

| Problem | POS Tagging Solution |
|----------|----------------------|
| “He was” could mean many things | Distinguish between “was angry” (state) and “was hit” (passive event). |
| “Said” vs “is said” | Detects active vs. passive voice. |
| “He looked” vs “looked at him” | Identifies if “he” is subject or object. |
| Counting verbs vs adjectives | Measures how often a character performs an action vs. being described. |

By applying POS tagging (and ideally, dependency parsing), we can:
- Measure grammatical agency quantitatively.
- Distinguish between active and passive constructions.
- Better understand narrative focus and character treatment.

-----------------------------------------
5. Conclusion
-----------------------------------------
Without grammatical awareness, bigram frequencies give only a superficial view of character behavior.
POS tagging, however, would allow us to identify action, emotion, and voice more precisely — turning a list of word pairs into meaningful literary insight.

Summary:
- Crude patterns: hint that “he” acts, “she” feels.
- Errors: no context, syntax, or agency direction.
- POS tagging: introduces grammatical depth, showing who does what — and how the narrative distributes agency.'''


'\n\n-----------------------------------------\n1. Overview\n-----------------------------------------\nThe analysis compares two books — Crime and Punishment and The Brothers Karamazov — by examining which words most frequently follow “he” and “she.” \nThis gives a crude sense of how male and female characters are described or portrayed in the narrative.\n\nExample data:\nCrime and Punishment (he): was, is, his, said, went, thought, asked...\nCrime and Punishment (she): was, is, said, cried, looked, seemed...\nThe Brothers Karamazov (he): was, is, said, went, thought, cried...\nThe Brothers Karamazov (she): was, is, said, cried, loved, looked...\n\n-----------------------------------------\n2. Character Agency and Narrative Voice\n-----------------------------------------\nCharacter agency refers to how active or passive characters are — whether they perform actions or have actions performed on them.\nNarrative voice refers to how the narrator describes and frames the characters.\n\nF